In [2]:
!pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 109.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 MB 112.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [xgboost]m1/2 [xgboost]


In [8]:
# =============================================================================
# Model 3 — XGBoost Regressor (complete)
# Task    : Predict Crime_Count based on Location and Date
# =============================================================================

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings("ignore")

print(f"XGBoost version: {xgb.__version__}")

# =============================================================================
# 1. LOAD DATA
# =============================================================================
df = pd.read_csv("./crime_per_capita_df_corrected.csv")

print(f"Dataset shape : {df.shape}")
print(f"Date range    : {df['Year'].min()} to {df['Year'].max()}")
print(f"Unique LSOAs  : {df['LSOA_Code'].nunique()}")

# =============================================================================
# 2. FEATURE SELECTION
# XGBoost is tree-based — no scaling needed
# Do NOT include Crime_Rate, Crime_per_Capita (derived from target = leakage)
# =============================================================================
FEATURES = [
    # --- Location ---
    "LSOA_Latitude",
    "LSOA_Longitude",
    "LSOA_Shape_Area",
    "LSOA_Shape_Length",

    # --- Date ---
    "Year",
    "Month",
    "month_sin",
    "month_cos",

    # --- Season flags ---
    "is_summer",
    "is_winter",
    "is_spring",
    "is_autumn",

    # --- Socioeconomic ---
    "Total_Population",
    "pop_density",
    "income_per_capita",
    "Income_Deprivation",

    # --- Weather ---
    "Max_Temperature_Celsius",
    "Min_Temperature_Celsius",
    "Rainfall_mm",
    "Sunshine_Hours",
    "Air_Frost_Days",

    # --- Lag & rolling (leak-free) ---
    "crime_lag_1m",
    "crime_lag_2m",
    "crime_lag_3m",
    "crime_lag_6m",
    "crime_rolling_3m_mean",
    "crime_rolling_3m_std",
    "crime_rolling_6m_mean",
    "crime_rolling_6m_std",
    "msoa_avg_crime",
]

TARGET = "Crime_Count"

missing = [f for f in FEATURES if f not in df.columns]
if missing:
    print(f"WARNING — missing columns: {missing}")
    FEATURES = [f for f in FEATURES if f in df.columns]

print(f"\nUsing {len(FEATURES)} features → target: '{TARGET}'")

# =============================================================================
# 3. TIME-BASED TRAIN / TEST SPLIT
# =============================================================================
train = df[df["Year"].isin([2020, 2021, 2022, 2023])].copy()
test  = df[df["Year"] == 2024].copy()

X_train = train[FEATURES]
y_train = train[TARGET]
X_test  = test[FEATURES]
y_test  = test[TARGET]

print(f"Train : {len(X_train):,} rows  |  Test : {len(X_test):,} rows")

# =============================================================================
# 4. EVALUATION HELPER
# =============================================================================
def evaluate(model_name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100

    print(f"\n{'='*55}")
    print(f"  {model_name}")
    print(f"{'='*55}")
    print(f"  MAE   : {mae:.4f}")
    print(f"  RMSE  : {rmse:.4f}")
    print(f"  R²    : {r2:.4f}")
    print(f"  MAPE  : {mape:.2f}%")

    return {"Model": model_name, "MAE": round(mae,4), "RMSE": round(rmse,4),
            "R2": round(r2,4), "MAPE": round(mape,2)}

results = []

# =============================================================================
# 5. BASELINE XGBOOST
# Sensible defaults — good starting point before tuning
# =============================================================================
print("\n--- Fitting Baseline XGBoost ---")

xgb_base = xgb.XGBRegressor(
    n_estimators     = 200,
    learning_rate    = 0.1,
    max_depth        = 6,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 1,
    reg_alpha        = 0,
    reg_lambda       = 1,
    objective        = "reg:squarederror",
    random_state     = 42,
    n_jobs           = -1,
    verbosity        = 0
)
xgb_base.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
preds_base = np.clip(xgb_base.predict(X_test), 0, None)
res_base   = evaluate("XGBoost — baseline", y_test, preds_base)
results.append({**res_base, "model_obj": xgb_base})

# =============================================================================
# 6. EARLY STOPPING
# Trains up to 1000 trees but stops when val score stops improving for 50 rounds
# Lower LR than baseline — slower but generalises better
# =============================================================================
print("\n--- XGBoost with Early Stopping (LR=0.05, up to 1000 trees) ---")

xgb_es = xgb.XGBRegressor(
    n_estimators          = 1000,
    learning_rate         = 0.05,
    max_depth             = 6,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    min_child_weight      = 3,
    reg_alpha             = 0.1,
    reg_lambda            = 1.0,
    objective             = "reg:squarederror",
    early_stopping_rounds = 50,
    random_state          = 42,
    n_jobs                = -1,
    verbosity             = 0
)
xgb_es.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=100)
best_iter = xgb_es.best_iteration
print(f"\nBest iteration: {best_iter}")
preds_es = np.clip(xgb_es.predict(X_test), 0, None)
res_es   = evaluate(f"XGBoost — early stopping (iter={best_iter})", y_test, preds_es)
results.append({**res_es, "model_obj": xgb_es})

# =============================================================================
# 7. TUNED VARIANTS
# Testing depth, regularisation, and learning rate combinations
# =============================================================================
print("\n--- Tuned XGBoost Variants ---")

# --- Shallow trees ---
print("\n  Fitting: shallow trees (depth=4)")
xgb_shallow = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=4,
    subsample=0.8, colsample_bytree=0.8,
    min_child_weight=3, reg_alpha=0.1, reg_lambda=1.0,
    objective="reg:squarederror", random_state=42, n_jobs=-1, verbosity=0
)
xgb_shallow.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
preds_shallow = np.clip(xgb_shallow.predict(X_test), 0, None)
res_shallow   = evaluate("XGBoost — shallow trees (depth=4)", y_test, preds_shallow)
results.append({**res_shallow, "model_obj": xgb_shallow})

# --- Deep trees ---
print("\n  Fitting: deep trees (depth=8)")
xgb_deep = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=8,
    subsample=0.8, colsample_bytree=0.8,
    min_child_weight=3, reg_alpha=0.1, reg_lambda=1.0,
    objective="reg:squarederror", random_state=42, n_jobs=-1, verbosity=0
)
xgb_deep.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
preds_deep = np.clip(xgb_deep.predict(X_test), 0, None)
res_deep   = evaluate("XGBoost — deep trees (depth=8)", y_test, preds_deep)
results.append({**res_deep, "model_obj": xgb_deep})

# --- Strong regularisation ---
print("\n  Fitting: strong regularisation")
xgb_reg = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    min_child_weight=5, reg_alpha=1.0, reg_lambda=5.0,
    objective="reg:squarederror", random_state=42, n_jobs=-1, verbosity=0
)
xgb_reg.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
preds_reg = np.clip(xgb_reg.predict(X_test), 0, None)
res_reg   = evaluate("XGBoost — strong regularisation", y_test, preds_reg)
results.append({**res_reg, "model_obj": xgb_reg})

# --- Slow LR (Config A) ---
# Very low LR forces gradual learning — often the best generalisation
# Early stopping with 100 round patience finds the true optimal iteration
print("\n  Fitting: slow LR=0.01 (up to 3000 trees, early stopping patience=100)")
xgb_slow = xgb.XGBRegressor(
    n_estimators=3000, learning_rate=0.01,
    max_depth=6, subsample=0.8, colsample_bytree=0.8,
    min_child_weight=3, reg_alpha=0.05, reg_lambda=1.0,
    early_stopping_rounds=100,
    objective="reg:squarederror", random_state=42, n_jobs=-1, verbosity=0
)
xgb_slow.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=200)
best_iter_slow = xgb_slow.best_iteration
print(f"\nBest iteration (slow LR): {best_iter_slow}")
preds_slow = np.clip(xgb_slow.predict(X_test), 0, None)
res_slow   = evaluate(f"XGBoost — slow LR=0.01 (iter={best_iter_slow})", y_test, preds_slow)
results.append({**res_slow, "model_obj": xgb_slow})

# --- Wide trees (Config B) ---
# Deeper trees + more subsampling noise = better spatial generalisation
print("\n  Fitting: wide trees (depth=7, LR=0.02, early stopping patience=100)")
xgb_wide = xgb.XGBRegressor(
    n_estimators=1000, learning_rate=0.02,
    max_depth=7, subsample=0.7, colsample_bytree=0.7,
    min_child_weight=5, reg_alpha=0.1, reg_lambda=2.0,
    early_stopping_rounds=100,
    objective="reg:squarederror", random_state=42, n_jobs=-1, verbosity=0
)
xgb_wide.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=200)
best_iter_wide = xgb_wide.best_iteration
print(f"\nBest iteration (wide): {best_iter_wide}")
preds_wide = np.clip(xgb_wide.predict(X_test), 0, None)
res_wide   = evaluate(f"XGBoost — wide trees depth=7 LR=0.02 (iter={best_iter_wide})", y_test, preds_wide)
results.append({**res_wide, "model_obj": xgb_wide})

# =============================================================================
# 8. PICK BEST MODEL
# =============================================================================
best     = min(results, key=lambda x: x["MAE"])
best_xgb = best["model_obj"]

print(f"\n{'='*55}")
print(f"Best XGBoost config : {best['Model']}")
print(f"MAE={best['MAE']}  RMSE={best['RMSE']}  R²={best['R2']}  MAPE={best['MAPE']}%")
print(f"{'='*55}")

# =============================================================================
# 9. FEATURE IMPORTANCE (by gain — most reliable)
# gain = average improvement in loss each time a feature is used in a split
# weight = how often a feature is used (less reliable on its own)
# =============================================================================
print("\n--- Feature Importance — top 15 (by gain) ---")

importance_gain = best_xgb.get_booster().get_score(importance_type="gain")
fi_df = pd.DataFrame({
    "Feature": list(importance_gain.keys()),
    "Gain"   : list(importance_gain.values())
}).sort_values("Gain", ascending=False)

print(fi_df.head(15).to_string(index=False))

lag_cols   = [f for f in FEATURES if "lag" in f or "rolling" in f or "msoa_avg" in f]
lag_gain   = fi_df[fi_df["Feature"].isin(lag_cols)]["Gain"].sum()
total_gain = fi_df["Gain"].sum()
print(f"\nLag/rolling features : {lag_gain/total_gain*100:.1f}% of total gain")
print(f"All other features   : {(total_gain-lag_gain)/total_gain*100:.1f}% of total gain")

# =============================================================================
# 10. LEARNING CURVE
# Train on increasing proportions of data — does more data still help?
# =============================================================================
print("\n--- Learning Curve ---")

for frac in [0.2, 0.4, 0.6, 0.8, 1.0]:
    n      = int(len(X_train) * frac)
    lc_mdl = xgb.XGBRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbosity=0
    )
    lc_mdl.fit(X_train.iloc[:n], y_train.iloc[:n])
    lc_mae = mean_absolute_error(y_test, np.clip(lc_mdl.predict(X_test), 0, None))
    print(f"  {int(frac*100):3d}% of train ({n:,} rows) → MAE: {lc_mae:.4f}")

# =============================================================================
# 11. RESIDUAL ANALYSIS
# Where does the best XGBoost model struggle most?
# =============================================================================
print("\n--- Residual Analysis ---")

best_preds = np.clip(best_xgb.predict(X_test), 0, None)
res_df = test[["LSOA_Code", "Year", "Month", TARGET]].copy()
res_df["Predicted"] = np.round(best_preds, 2)
res_df["Residual"]  = res_df[TARGET] - res_df["Predicted"]
res_df["Abs_Error"] = res_df["Residual"].abs()
res_df["Pct_Error"] = res_df["Abs_Error"] / (res_df[TARGET] + 1e-9) * 100

# Error by crime count bucket — shows if model struggles at low or high counts
res_df["Crime_Bucket"] = pd.cut(
    res_df[TARGET], bins=[0, 5, 15, 30, 50, 9999],
    labels=["0–5", "6–15", "16–30", "31–50", "50+"]
)
bucket_err = (res_df.groupby("Crime_Bucket", observed=True)["Abs_Error"]
              .agg(["mean", "count"])
              .rename(columns={"mean": "Avg_MAE", "count": "N_rows"}))
print("\nError by crime count bucket:")
print(bucket_err.to_string())

# Worst LSOAs by average prediction error
worst = (res_df.groupby("LSOA_Code")["Abs_Error"]
         .mean().sort_values(ascending=False).head(8))
print("\nTop 8 hardest LSOAs:")
print(worst.to_string())

# Error pattern by month
monthly = (res_df.groupby("Month")["Abs_Error"]
           .mean().reset_index()
           .rename(columns={"Abs_Error": "Avg_MAE"}))
print("\nAverage error by month:")
print(monthly.to_string(index=False))

# =============================================================================
# 12. SAMPLE PREDICTIONS
# =============================================================================
print("\n--- Sample Predictions vs Actual (first 10 test rows) ---")
print(res_df.head(10)[
    ["LSOA_Code","Year","Month",TARGET,"Predicted","Residual"]
].to_string(index=False))

# =============================================================================
# 13. SAVE TO MODEL COMPARISON FILE
# =============================================================================
new_rows = pd.DataFrame([
    {k: v for k, v in r.items() if k != "model_obj"}
    for r in results
])

try:
    existing = pd.read_csv("model_comparison.csv")
    combined = pd.concat([existing, new_rows], ignore_index=True)
except FileNotFoundError:
    combined = new_rows

combined.to_csv("model_comparison.csv", index=False)

print("\n\n--- Final model_comparison.csv ---")
print(combined[["Model","MAE","RMSE","R2","MAPE"]].to_string(index=False))

XGBoost version: 3.2.0
Dataset shape : (304336, 39)
Date range    : 2020 to 2024
Unique LSOAs  : 12415

Using 30 features → target: 'Crime_Count'
Train : 241,710 rows  |  Test : 62,626 rows

--- Fitting Baseline XGBoost ---

  XGBoost — baseline
  MAE   : 4.5809
  RMSE  : 9.3732
  R²    : 0.9326
  MAPE  : 43.90%

--- XGBoost with Early Stopping (LR=0.05, up to 1000 trees) ---
[0]	validation_0-rmse:34.68619
[100]	validation_0-rmse:10.02907
[200]	validation_0-rmse:9.76893
[300]	validation_0-rmse:9.73519
[400]	validation_0-rmse:9.72073
[421]	validation_0-rmse:9.72616

Best iteration: 371

  XGBoost — early stopping (iter=371)
  MAE   : 4.6011
  RMSE  : 9.7131
  R²    : 0.9276
  MAPE  : 43.10%

--- Tuned XGBoost Variants ---

  Fitting: shallow trees (depth=4)

  XGBoost — shallow trees (depth=4)
  MAE   : 4.6680
  RMSE  : 10.2395
  R²    : 0.9195
  MAPE  : 44.59%

  Fitting: deep trees (depth=8)

  XGBoost — deep trees (depth=8)
  MAE   : 4.5636
  RMSE  : 9.4262
  R²    : 0.9318
  MAPE  :